In [1]:
!pip install pandas datasets spacy
!python -m spacy download ru_core_news_sm

     ---------------------------------------- 0.0/15.3 MB ? eta -:--:--
     ---------------------------------------- 0.0/15.3 MB ? eta -:--:--
     ---------------------------------------- 0.0/15.3 MB ? eta -:--:--
     ---------------------------------------- 0.0/15.3 MB ? eta -:--:--
     ---------------------------------------- 0.0/15.3 MB ? eta -:--:--
     ---------------------------------------- 0.0/15.3 MB ? eta -:--:--
      --------------------------------------- 0.3/15.3 MB ? eta -:--:--
      --------------------------------------- 0.3/15.3 MB ? eta -:--:--
      --------------------------------------- 0.3/15.3 MB ? eta -:--:--
      --------------------------------------- 0.3/15.3 MB ? eta -:--:--
     - ------------------------------------- 0.5/15.3 MB 391.8 kB/s eta 0:00:38
     - ------------------------------------- 0.5/15.3 MB 391.8 kB/s eta 0:00:38
     -- ------------------------------------ 0.8/15.3 MB 495.4 kB/s eta 0:00:30
     -- --------------------------------

In [1]:
from difflib import SequenceMatcher

import pandas as pd
import spacy
from datasets import load_dataset

c:\Users\Pauline\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
nlp = spacy.load("ru_core_news_sm")

print("spaCy model loaded")

spaCy model loaded


In [3]:
# -----------------------------
def clean_text(text: str) -> str:
    doc = nlp(text)

    tokens = [
        token.lemma_.lower()
        for token in doc
        if not (
            token.is_stop
            or token.is_punct
            or token.is_space
            or token.pos_ in {"PUNCT", "SYM"}
        )
    ]

    return " ".join(tokens)


# -----------------------------
# Извлечение различий
# -----------------------------
def extract_diff(toxic: str, neutral: str) -> str:
    toxic_tokens = toxic.split()
    neutral_tokens = neutral.split()

    matcher = SequenceMatcher(
        None,
        toxic_tokens,
        neutral_tokens
    )

    diffs = []

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():

        if tag == "equal":
            continue

        toxic_part = " ".join(toxic_tokens[i1:i2])
        neutral_part = " ".join(neutral_tokens[j1:j2])

        toxic_part = clean_text(toxic_part)
        neutral_part = clean_text(neutral_part)

        if not toxic_part and not neutral_part:
            continue

        if tag == "delete":
            diffs.append(f"DELETE[{toxic_part}]")

        elif tag == "insert":
            diffs.append(f"INSERT[{neutral_part}]")

        elif tag == "replace":
            diffs.append(
                f"REPLACE[{toxic_part}] -> [{neutral_part}]"
            )

    return " ; ".join(diffs)


# -----------------------------
# Загрузка датасета
# -----------------------------
dataset = load_dataset(
    "s-nlp/ru_paradetox",
    split="train"
)

# -----------------------------
# Формирование результата
# -----------------------------
rows = []

for sample in dataset:

    toxic = sample["ru_toxic_comment"]
    neutral = sample["ru_neutral_comment"]

    diff = extract_diff(toxic, neutral)

    rows.append({
        "toxic_comment": toxic,
        "neutral_comment": neutral,
        "difference": diff
    })

result_df = pd.DataFrame(rows)

# -----------------------------
# Сохранение
# -----------------------------
result_df.to_csv(
    "ru_paradetox_differences.csv",
    index=False,
    encoding="utf-8-sig"
)

print(result_df.head(10))

KeyboardInterrupt: 